In [2]:
from pathlib import Path
import pandas as pd

file_path = Path(__file__).parent / 'combined_hesitancy_sdoh_cumulative_death_rate.txt'
combined_df = pd.read_csv(file_path, sep='\t')
output_path = file_path.with_suffix('.csv')
combined_df.to_csv(output_path, index=False)

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print(combined_df.head())
print(combined_df.info()) 
print(combined_df.columns)

  FIPS,County,State,Population,Hesitancy_Rate,SDOH_Aged 65 years or older,SDOH_Crowding,SDOH_Housing cost burden,SDOH_No broadband,SDOH_No high school diploma,SDOH_Poverty,SDOH_Racial or ethnic minority status,SDOH_Single-parent households,SDOH_Unemployment,2020-01-20/2020-01-26,2020-01-27/2020-02-02,2020-02-03/2020-02-09,2020-02-10/2020-02-16,2020-02-17/2020-02-23,2020-02-24/2020-03-01,2020-03-02/2020-03-08,2020-03-09/2020-03-15,2020-03-16/2020-03-22,2020-03-23/2020-03-29,2020-03-30/2020-04-05,2020-04-06/2020-04-12,2020-04-13/2020-04-19,2020-04-20/2020-04-26,2020-04-27/2020-05-03,2020-05-04/2020-05-10,2020-05-11/2020-05-17,2020-05-18/2020-05-24,2020-05-25/2020-05-31,2020-06-01/2020-06-07,2020-06-08/2020-06-14,2020-06-15/2020-06-21,2020-06-22/2020-06-28,2020-06-29/2020-07-05,2020-07-06/2020-07-12,2020-07-13/2020-07-19,2020-07-20/2020-07-26,2020-07-27/2020-08-02,2020-08-03/2020-08-09,2020-08-10/2020-08-16,2020-08-17/2020-08-23,2020-08-24/2020-08-30,2020-08-31/2020-09-06,2020-09-07/2020-

### Checking for Missingness

In [9]:
missing_pct = combined_df.isna().mean() * 100
missing_pct = missing_pct.sort_values(ascending=False)

print("Percent missing in each column:")
print(missing_pct)

Percent missing in each column:
FIPS                     0.0
2022-05-02/2022-05-08    0.0
2021-12-13/2021-12-19    0.0
2021-12-20/2021-12-26    0.0
2021-12-27/2022-01-02    0.0
                        ... 
2020-12-14/2020-12-20    0.0
2020-12-21/2020-12-27    0.0
2020-12-28/2021-01-03    0.0
2021-01-04/2021-01-10    0.0
2023-03-06/2023-03-12    0.0
Length: 178, dtype: float64


### Duplicaitons

In [5]:
# Split the single column into multiple columns based on commas
columns = combined_df.columns[0].split(',')
combined_df = combined_df.iloc[:, 0].str.split(',', expand=True)
combined_df.columns = columns

# Check for duplications in the first 20 columns
for col in combined_df.columns[:20]:
    print(f"{col}: {combined_df[col].duplicated().sum()} duplicates")

FIPS: 0 duplicates
County: 1303 duplicates
State: 3089 duplicates
Population: 54 duplicates
Hesitancy_Rate: 3113 duplicates
SDOH_Aged 65 years or older: 2863 duplicates
SDOH_Crowding: 3014 duplicates
SDOH_Housing cost burden: 2851 duplicates
SDOH_No broadband: 2745 duplicates
SDOH_No high school diploma: 2840 duplicates
SDOH_Poverty: 2727 duplicates
SDOH_Racial or ethnic minority status: 2409 duplicates
SDOH_Single-parent households: 2988 duplicates
SDOH_Unemployment: 2980 duplicates
2020-01-20/2020-01-26: 3138 duplicates
2020-01-27/2020-02-02: 3138 duplicates
2020-02-03/2020-02-09: 3138 duplicates
2020-02-10/2020-02-16: 3138 duplicates
2020-02-17/2020-02-23: 3138 duplicates
2020-02-24/2020-03-01: 3137 duplicates


These duplicates are irrelevant because it's qualitative data with fixed answers

### Check for Negative Death Counts

In [6]:
date_cols = combined_df.columns[combined_df.columns.str.contains(r'\d{4}-\d{2}-\d{2}/\d{4}-\d{2}-\d{2}')]

date_data = combined_df[date_cols].apply(pd.to_numeric, errors='coerce')
positive_death_mask = date_data > 0

positive_by_state = (
    combined_df.assign(has_positive_deaths=positive_death_mask.any(axis=1))
    .groupby('State')['has_positive_deaths']
    .any()
)

print("Any positive death counts overall:", positive_death_mask.any().any())
print("States with any positive death counts:")
print(positive_by_state[positive_by_state].sort_index())

Any positive death counts overall: True
States with any positive death counts:
State
Alabama                 True
Alaska                  True
Arizona                 True
Arkansas                True
California              True
Colorado                True
Connecticut             True
Delaware                True
District of Columbia    True
Florida                 True
Georgia                 True
Hawaii                  True
Idaho                   True
Illinois                True
Indiana                 True
Iowa                    True
Kansas                  True
Kentucky                True
Louisiana               True
Maine                   True
Maryland                True
Massachusetts           True
Michigan                True
Minnesota               True
Mississippi             True
Missouri                True
Montana                 True
Nebraska                True
Nevada                  True
New Hampshire           True
New Jersey              True
New Mexico      

### Are death rates accumulating over time?

In [7]:
# Check if death rates are accumulating over time
# Calculate the differences between consecutive time periods for each row
death_rate_differences = date_data.diff(axis=1)

# Check if all differences are non-negative (indicating accumulation)
is_accumulating = (death_rate_differences >= 0).all().all()

print(f"Are death rates accumulating over time? {is_accumulating}")
print(f"\nNumber of negative differences (should be 0 if accumulating): {(death_rate_differences < 0).sum().sum()}")

# Show a few examples of rate changes
print("\nSample of death rate changes (first 5 rows, columns 10-15):")
print(death_rate_differences.iloc[:5, 10:15])

Are death rates accumulating over time? False

Number of negative differences (should be 0 if accumulating): 3649

Sample of death rate changes (first 5 rows, columns 10-15):
   2020-03-30/2020-04-05  2020-04-06/2020-04-12  2020-04-13/2020-04-19  \
0                    0.0               1.789901               1.789901   
1                    0.0               0.000000               0.447960   
2                    0.0               0.000000               0.000000   
3                    0.0               0.000000               0.000000   
4                    0.0               0.000000               0.000000   

   2020-04-20/2020-04-26  2020-04-27/2020-05-03  
0                0.00000               1.789901  
1                0.44796               0.447960  
2                0.00000               4.050879  
3                0.00000               0.000000  
4                0.00000               0.000000  


In [8]:
# Show only negative differences
negative_diffs = death_rate_differences[death_rate_differences < 0]

if negative_diffs.notna().any().any():
    print("Negative differences found:")
    print(negative_diffs.dropna(how='all').dropna(axis=1, how='all'))
else:
    print("No negative differences - death rates are accumulating as expected")

Negative differences found:
      2020-03-09/2020-03-15  2020-03-23/2020-03-29  2020-03-30/2020-04-05  \
0                       NaN                    NaN                    NaN   
1                       NaN                    NaN                    NaN   
2                       NaN                    NaN                    NaN   
3                       NaN                    NaN                    NaN   
4                       NaN                    NaN                    NaN   
...                     ...                    ...                    ...   
3115                    NaN                    NaN                    NaN   
3116                    NaN                    NaN                    NaN   
3121                    NaN                    NaN                    NaN   
3129                    NaN                    NaN                    NaN   
3133                    NaN                    NaN                    NaN   

      2020-04-06/2020-04-12  2020-04-13/2020-04